# 01  Data loading and storm-level summary

This notebook loads the TropiCycloneNet Data1D files, converts the raw track observations into a clean timestep-level dataset, and builds one row per storm.

The output is used in later notebooks to analyse high-intensity storm share, rapid intensification, and movement speed metrics across cyclone basins.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from load_data1d import (
    BASINS,
    DATA1D_ROOT,
    build_storm_summary,
    load_all_basins,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("data root exists:", DATA1D_ROOT.exists())
print("processed output directory exists:", PROCESSED_DIR.exists())

data root exists: True
processed output directory exists: True


Loading data and building storm summary.

In [2]:
tracks = load_all_basins()
summary = build_storm_summary(tracks)

print(f"per-timestep rows: {len(tracks):,}")
print(f"per-storm rows:    {len(summary):,}")
summary.head(3)

Loading WP... 59,183 timesteps from 1,799 storms
Loading EP... 15,971 timesteps from 497 storms
Loading NA... 18,395 timesteps from 517 storms
Loading SI... 19,819 timesteps from 557 storms
Loading SP... 9,565 timesteps from 308 storms
per-timestep rows: 122,933
per-storm rows:    3,678


,storm_id,basin,year,storm_name,split,peak_wind_ms,intensity_class,is_major_hurricane,is_intense_or_above,is_extreme,experienced_RI,mean_translation_kmh,genesis_lat,genesis_lon,lifetime_hours,n_observations,decade
0,WP_WP1950BSTBILLIE,WP,1950,Billie,train,35.000,Hurricane (low),False,False,False,True,30.802,13.000,150.800,162.000,28,1950
1,WP_WP1950BSTDELILAH,WP,1950,Delilah,train,30.000,Tropical storm,False,False,False,True,25.927,6.500,138.300,168.000,29,1950
2,WP_WP1950BSTDORIS,WP,1950,Doris,train,70.000,Intense,True,True,False,True,29.106,8.600,150.800,234.000,40,1950


### Sanity check with known historical storms

Before using the storm-level summary for analysis, I check two well-known storms to confirm that storm names, years, basins, peak wind values, intensity labels, and rapid-intensification flags are being parsed consistently.

In [3]:
known_storms = [
    {
        "name": "DORA",
        "basin": "EP",
        "year": 2017,
        "expected_peak_range": (40, 50),
        "expected_class": "Hurricane (mid)"
    },
    {
        "name": "HAIYAN",
        "basin": "WP",
        "year": 2013,
        "expected_peak_range": (75, 90),
        "expected_class": "Extreme"
    }
]

check_rows = []

for storm in known_storms:
    match = summary[
        summary["storm_name"].str.contains(storm["name"], case=False, na=False)
        & (summary["year"] == storm["year"])
        & (summary["basin"] == storm["basin"])
    ]

    if match.empty:
        check_rows.append({
            "storm": storm["name"],
            "found": False,
            "peak_wind_ms": np.nan,
            "intensity_class": None,
            "experienced_RI": None,
            "peak_range_check": False,
            "class_check": False
        })
    else:
        row = match.sort_values("peak_wind_ms", ascending=False).iloc[0]
        low, high = storm["expected_peak_range"]

        check_rows.append({
            "storm": storm["name"],
            "found": True,
            "peak_wind_ms": round(row["peak_wind_ms"], 1),
            "intensity_class": row["intensity_class"],
            "experienced_RI": row["experienced_RI"],
            "peak_range_check": low <= row["peak_wind_ms"] <= high,
            "class_check": row["intensity_class"] == storm["expected_class"]
        })

pd.DataFrame(check_rows)

,storm,found,peak_wind_ms,intensity_class,experienced_RI,peak_range_check,class_check
0,DORA,True,46.300,Hurricane (mid),True,True,True
1,HAIYAN,True,78.000,Extreme,True,True,True


### Coverage check by basin and decade

I check whether each basin has enough storm coverage over time. This helps identify sparse periods.The 2020 column is a partial decade because the dataset ends in 2023. 

In [4]:
decade_counts = (
    summary.assign(decade=(summary["year"] // 10 * 10).astype(int))
    .groupby(["basin", "decade"])
    .size()
    .unstack("decade", fill_value=0)
    .reindex(list(BASINS))
)

print("missing years per basin within 1980-2023:")
for b in BASINS:
    have = set(summary.loc[summary["basin"] == b, "year"].unique())
    missing = sorted(set(range(1980, 2024)) - have)
    if not missing:
        print(f"  {b}: complete")
    elif len(missing) <= 12:
        print(f"  {b}: {len(missing)}/44 years missing {missing}")
    else:
        print(f"  {b}: {len(missing)}/44 years missing")

decade_counts

missing years per basin within 1980-2023:
  WP: complete
  EP: 8/44 years missing [1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987]
  NA: complete
  SI: complete
  SP: complete


decade,1950,1960,1970,1980,1990,2000,2010,2020
basin,,,,,,,,
WP,195,293,258,258,261,230,242,62
EP,0,0,0,26,125,133,155,58
NA,0,24,30,69,93,113,119,69
SI,0,0,31,116,109,135,123,43
SP,0,0,22,73,76,60,62,15


The check shows that the Eastern Pacific basin is missing 1980–1987, while the selected basins are complete from 1988 onward. I therefore start the main analysis in 1988 to keep it consistent annually across basins.

This is also a practical data-quality choice. More recent records are generally better observed, while earlier historical records are more likely to miss weaker or shorter-lived storms.

1980 decade will also be partial after the cut including 1988 and 1989 only.

In [5]:
#Restrict to 1988-2023, NI dropped in load step.
ANALYSIS_BASINS = ['WP', 'EP', 'NA', 'SI', 'SP']
START_YEAR = 1988

summary_clean = summary[
    summary['basin'].isin(ANALYSIS_BASINS)
    & (summary['year'] >= START_YEAR)
    & (summary['year'] <= 2023)
].copy()

tracks_clean = tracks[
    tracks['basin'].isin(ANALYSIS_BASINS)
    & (tracks['year'] >= START_YEAR)
    & (tracks['year'] <= 2023)
].copy()

print(f"Analysis dataset: {len(summary_clean):,} storms across {len(ANALYSIS_BASINS)} basins, 1988-2023")
print(f"Per-timestep rows: {len(tracks_clean):,}")
print(f"\nStorms per basin per decade:")
print(pd.crosstab(summary_clean['basin'], summary_clean['decade']).reindex(ANALYSIS_BASINS))

Analysis dataset: 2,424 storms across 5 basins, 1988-2023
Per-timestep rows: 80,688

Storms per basin per decade:
decade  1980  1990  2000  2010  2020
basin                               
WP        56   261   230   242    62
EP        26   125   133   155    58
NA        17    93   113   119    69
SI        25   109   135   123    43
SP        17    76    60    62    15


In [6]:
# Save analysis-ready dataset for notebooks
summary_clean.to_parquet(PROCESSED_DIR / "storms_analysis.parquet")
tracks_clean.to_parquet(PROCESSED_DIR / "tracks_analysis.parquet")

print("Saved processed data")
print(f"  storms_analysis.parquet: {len(summary_clean):,} rows")
print(f"  tracks_analysis.parquet: {len(tracks_clean):,} rows")

Saved processed data
  storms_analysis.parquet: 2,424 rows
  tracks_analysis.parquet: 80,688 rows
